# Clinical Gemma LoRA pilot on the frozen prepared dataset

Run after preparation and the prepared baseline. Default mode verifies data/labels and runs a finite forward/backward check **without optimizer steps**. It does not train until RUN_TRAINING is explicitly True. Model loading and the backward check need a suitable GPU for practical execution.

Upload the saved prepared dataset and the numbered baseline result folder to the **same remote runtime** as this notebook. Paths refer to that runtime. This notebook consumes the saved token IDs and assistant-only labels directly: no reformatting, truncation, packing, extra BOS tokens, or TRL preprocessing. It pins the original model revision and baseline contract. Do not use the older 512-token training notebook.

A 60-step pilot is an initial learning check, not a claim of convergence or clinical usefulness. The 10 validation cases are the same development comparison set used by the baseline, not an untouched test set. Training logs, label previews, and outputs contain clinical text. No paid APIs or external tracking are used.


In [16]:
%pip install --progress-bar on "transformers==4.57.6" "datasets>=3,<5" "peft>=0.17,<0.19" "accelerate>=1,<2" pandas


In [17]:
import os, json, hashlib, importlib.metadata, random
from pathlib import Path
from getpass import getpass
import torch
from datasets import load_from_disk
from huggingface_hub import get_token
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, set_seed
from peft import LoraConfig, TaskType, get_peft_model

PREPARED_ROOT = Path("prepared_clinical_dataset")
BASELINE_DIR = Path("baseline_evaluation_diagnostic_10/900825a06d38a006_353c8393d011")
RUN_TRAINING = True  # First run checks labels/loss/gradients; no optimizer updates.
CHECK_BACKWARD = True
MAX_STEPS = 166
GRADIENT_ACCUMULATION_STEPS = 8
LEARNING_RATE = 2e-4
OUTPUT_ROOT = Path("clinical_notebook_pilot")
RUN_NAME = "pilot_02"  # Choose a new name for each training attempt.
assert MAX_STEPS > 0 and GRADIENT_ACCUMULATION_STEPS > 0 and LEARNING_RATE > 0

def digest(value):
    return hashlib.sha256(json.dumps(value, ensure_ascii=False, sort_keys=True).encode()).hexdigest()

def runtime_path(path, sentinel):
    path = Path(path).expanduser()
    candidates = [path]
    if not path.is_absolute():
        candidates += [parent/path for parent in Path.cwd().parents]
        candidates.append(Path("/content")/path)
    for candidate in candidates:
        if (candidate/sentinel).is_file():
            return candidate.resolve()
    raise FileNotFoundError(f"Kernel working directory: {Path.cwd()}. Could not find {sentinel} in {path}. "
                            "Upload the complete saved folder to this runtime and correct the configured path.")

print("Kernel working directory:", Path.cwd(), flush=True)
PREPARED_ROOT = runtime_path(PREPARED_ROOT, "latest.json")
BASELINE_DIR = runtime_path(BASELINE_DIR, "comparison_contract.json")
contract = json.loads((BASELINE_DIR/"comparison_contract.json").read_text(encoding="utf-8"))
version = contract["dataset_version"]  # Never select a different dataset via latest.json.
if not isinstance(version, str) or Path(version).name != version or version in (".", ".."):
    raise ValueError("Invalid dataset version")
prepared_dir = PREPARED_ROOT/version
manifest = json.loads((prepared_dir/"manifest.json").read_text(encoding="utf-8"))
prepared = load_from_disk(str(prepared_dir/"dataset"))
for split, sha in manifest["split_sha256"].items():
    assert digest(prepared[split].to_list()) == sha, f"Prepared {split} changed"
assert manifest["split_sha256"]["validation"] == contract["validation_sha256"]
assert list(prepared["validation"]["case_id"]) == contract["case_ids"]
assert digest(list(prepared["validation"]["prompt_text"])) == contract["prompt_sha256"]
assert manifest["config"]["model"] == contract["model"]
assert manifest["config"]["model_revision"] == contract["base_model_revision"]
print("Frozen dataset and baseline contract verified:", version, manifest["counts"], flush=True)

output_dir = OUTPUT_ROOT / version / (RUN_NAME if RUN_TRAINING else RUN_NAME + "_checks")
if RUN_TRAINING and output_dir.exists() and any(output_dir.iterdir()):
    raise FileExistsError("Training output exists; use a new RUN_NAME. Automatic training resume is not enabled.")
output_dir.mkdir(parents=True, exist_ok=True)
(output_dir/"comparison_contract.json").write_text(json.dumps(contract, indent=2), encoding="utf-8")

token = os.environ.get("HF_TOKEN") or get_token()
if not token:
    print("Enter your HF token in the VS Code input prompt.", flush=True)
    token = getpass("Hugging Face read token: ").strip()
if not token:
    raise RuntimeError("A token with Gemma access is required")
os.environ["HF_TOKEN"] = token
set_seed(contract["seed"])
tokenizer = AutoTokenizer.from_pretrained(contract["model"], revision=contract["base_model_revision"], token=token)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token


Kernel working directory: /content
Frozen dataset and baseline contract verified: 900825a06d38a006 {'train': 1325, 'validation': 10}


In [18]:
def pad_rows(features, pad_id):
    width = max(len(row["input_ids"]) for row in features)
    batch = {key: [] for key in ("input_ids", "attention_mask", "labels")}
    for row in features:
        padding = width-len(row["input_ids"])
        batch["input_ids"].append(list(row["input_ids"]) + [pad_id]*padding)
        batch["attention_mask"].append(list(row["attention_mask"]) + [0]*padding)
        batch["labels"].append(list(row["labels"]) + [-100]*padding)
    return batch

def collate(features):
    return {key: torch.tensor(value, dtype=torch.long)
            for key, value in pad_rows(features, tokenizer.pad_token_id).items()}

# Verify every saved row before touching model weights.
for split in ("train", "validation"):
    for row in prepared[split]:
        ids, labels, mask = row["input_ids"], row["labels"], row["attention_mask"]
        first = row["prompt_tokens"]
        assert len(ids) == len(labels) == len(mask) == row["total_tokens"]
        assert 0 < first < len(ids) <= manifest["config"]["max_sequence_length"]
        assert all(label == -100 for label in labels[:first])
        assert labels[first:] == ids[first:] and all(v == 1 for v in mask)
        decoded = tokenizer.decode(ids[first:], skip_special_tokens=False)
        assert decoded.startswith(row["reference"]), f"{row['case_id']}: supervised answer differs from reference"

mock = [{"input_ids":[1,2,3],"attention_mask":[1,1,1],"labels":[-100,2,3]},
        {"input_ids":[4,5],"attention_mask":[1,1],"labels":[-100,5]}]
padded = pad_rows(mock, 0)
assert padded["labels"][1] == [-100,5,-100]
assert padded["attention_mask"][1] == [1,1,0]
assert padded["input_ids"][1] == [4,5,0]

# Preview actual collated labels, including closing turn tokens.
previews = []
for row in list(prepared["train"].select(range(min(3,len(prepared["train"]))))):
    batch = collate([row])
    active = batch["labels"][0] != -100
    previews.append({"case_id": row["case_id"], "prompt_tokens": row["prompt_tokens"],
        "total_tokens": row["total_tokens"], "supervised_tokens": int(active.sum()),
        "decoded_labels": tokenizer.decode(batch["labels"][0][active].tolist(), skip_special_tokens=False)})
(output_dir/"label_checks.json").write_text(json.dumps(previews, ensure_ascii=False, indent=2), encoding="utf-8")
print("All saved labels verified; prompt and padding labels are masked.")
for row in previews:
    print(row["case_id"], "supervised:", row["supervised_tokens"],
          "start:", repr(row["decoded_labels"][:100]), "end:", repr(row["decoded_labels"][-70:]))


All saved labels verified; prompt and padding labels are masked.
train:0 supervised: 578 start: '{"visit motivation":"Injuries to the left hemithorax and a transfixing laceration in the left arm ca' end: '":"None","follow up":"None","discharge summary":"None"}}<end_of_turn>\n'
train:1 supervised: 657 start: '{"visit motivation":"Intermittent abdominal pain and anal bleeding","admission":[{"reason":"None","d' end: '":"None","follow up":"None","discharge summary":"None"}}<end_of_turn>\n'
train:2 supervised: 900 start: '{"visit motivation":"Lesion on the left foot","admission":[{"reason":"Lesion arising from the deep s' end: '":"None","follow up":"None","discharge summary":"None"}}<end_of_turn>\n'


In [19]:
if not torch.cuda.is_available():
    raise RuntimeError("Connect a GPU runtime for the model-backed pilot checks.")
# FP32 matches the completed baseline and avoids FP16 failure on T4.
dtype = torch.float32
print("Loading original model for LoRA checks:", torch.cuda.get_device_name(0), dtype, flush=True)
model = AutoModelForCausalLM.from_pretrained(contract["model"], revision=contract["base_model_revision"],
                                            token=token, dtype=dtype).to("cuda")
model.config.use_cache = False
model = get_peft_model(model, LoraConfig(
    task_type=TaskType.CAUSAL_LM, r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]))
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model.print_trainable_parameters()

# Check the longest prepared training row, not just an unusually short example.
longest = max(range(len(prepared["train"])), key=lambda i: prepared["train"][i]["total_tokens"])
row = prepared["train"][longest]
batch = {key: value.to(model.device) for key, value in collate([row]).items()}
torch.cuda.reset_peak_memory_stats()
model.eval()
with torch.no_grad():
    loss = model(**batch).loss
if not torch.isfinite(loss):
    raise RuntimeError("Non-finite initial loss; do not train")
check = {"case_id": row["case_id"], "total_tokens": row["total_tokens"],
         "initial_loss": float(loss.item()), "backward_checked": False}
del loss
if CHECK_BACKWARD:
    model.train()
    model.zero_grad(set_to_none=True)
    loss = model(**batch).loss
    if not torch.isfinite(loss):
        raise RuntimeError("Non-finite training-mode loss; do not train")
    loss.backward()  # Memory/gradient check only: no optimizer update.
    grads = [p.grad for p in model.parameters() if p.requires_grad and p.grad is not None]
    assert grads and all(bool(torch.isfinite(g).all()) for g in grads), "Missing or non-finite gradients"
    check.update(backward_checked=True, finite_gradients=True)
    model.zero_grad(set_to_none=True)
    del grads, loss
check["peak_allocated_gib"] = torch.cuda.max_memory_allocated()/1024**3
check["gpu"] = torch.cuda.get_device_name(0)
check["dtype"] = str(dtype)
del batch
torch.cuda.empty_cache()
(output_dir/"model_checks.json").write_text(json.dumps(check, indent=2), encoding="utf-8")
print(json.dumps(check, indent=2))
print("No optimizer updates have been made.")


Loading original model for LoRA checks: Tesla T4 torch.float32
trainable params: 3,796,992 || all params: 271,895,168 || trainable%: 1.3965
{
  "case_id": "train:64",
  "total_tokens": 1949,
  "initial_loss": 1.725236177444458,
  "backward_checked": true,
  "finite_gradients": true,
  "peak_allocated_gib": 7.882281303405762,
  "gpu": "Tesla T4",
  "dtype": "torch.float32"
}
No optimizer updates have been made.


In [20]:
run_config = {"dataset_version": version, "comparison_contract_sha256": digest(contract),
    "train_sha256": manifest["split_sha256"]["train"], "model": contract["model"],
    "base_model_revision": contract["base_model_revision"], "seed": contract["seed"],
    "run_training": RUN_TRAINING, "max_steps": MAX_STEPS, "batch_size": 1,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS, "learning_rate": LEARNING_RATE,
    "dtype": "float32", "loss": "saved assistant-only labels", "packing": False,
    "lora_rank": 16, "lora_alpha": 32, "lora_dropout": 0.05,
    "versions": {name: importlib.metadata.version(name) for name in (
        "torch", "transformers", "datasets", "peft", "accelerate")}}
(output_dir/"run_config.json").write_text(json.dumps(run_config, indent=2), encoding="utf-8")

if not RUN_TRAINING:
    print("Checks complete. Review model_checks.json and label_checks.json before enabling RUN_TRAINING.")
else:
    if not check["backward_checked"]:
        raise RuntimeError("Enable CHECK_BACKWARD and pass it before training")
    set_seed(contract["seed"])
    # Drop text/metadata columns: Trainer must receive only the verified tensors.
    columns = ["input_ids", "attention_mask", "labels"]
    train_data = prepared["train"].select_columns(columns)
    eval_data = prepared["validation"].select_columns(columns)
    args = TrainingArguments(
        output_dir=str(output_dir), per_device_train_batch_size=1, per_device_eval_batch_size=1,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        max_steps=MAX_STEPS, learning_rate=LEARNING_RATE, warmup_ratio=0.1,
        gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},
        bf16=False, fp16=False, logging_steps=1, logging_nan_inf_filter=False,
        save_strategy="no", report_to="none", seed=contract["seed"], data_seed=contract["seed"],
        label_names=["labels"], remove_unused_columns=False)

    class FiniteLossTrainer(Trainer):
        def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
            result = super().compute_loss(model, inputs, return_outputs=return_outputs, **kwargs)
            loss = result[0] if return_outputs else result
            if not torch.isfinite(loss).all():
                raise RuntimeError("Non-finite loss; stopping training")
            return result

    trainer = FiniteLossTrainer(model=model, args=args, data_collator=collate,
                               train_dataset=train_data, eval_dataset=eval_data)
    result = trainer.train()
    metrics = {"train": result.metrics, "validation": trainer.evaluate()}
    trainer.save_model(str(output_dir/"adapter"))
    tokenizer.save_pretrained(output_dir/"adapter")
    (output_dir/"metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
    (output_dir/"training_log.json").write_text(json.dumps(trainer.state.log_history, indent=2), encoding="utf-8")
    print("Saved adapter:", (output_dir/"adapter").resolve())
    print("Evaluate this adapter against the saved baseline contract; do not generate a new selection.")


Step,Training Loss
1,2.017700
2,1.943400
3,2.070000
4,1.871800
5,1.821400
6,1.604500
7,1.520500
8,1.408300
9,1.364800
10,1.298400


Saved adapter: /content/clinical_notebook_pilot/900825a06d38a006/pilot_02/adapter
Evaluate this adapter against the saved baseline contract; do not generate a new selection.


In [21]:
import shutil
archive = shutil.make_archive(str(output_dir.resolve()), "zip", root_dir=output_dir.parent,
                             base_dir=output_dir.name)
print("Download pilot/check artifacts:", archive)
# Browser Colab only: from google.colab import files; files.download(archive)


Download pilot/check artifacts: /content/clinical_notebook_pilot/900825a06d38a006/pilot_02.zip


If the longest-row backward check runs out of memory, report that error and the GPU type before changing the task or data. It checks batch-size-one memory, but optimizer state and evaluation can still add memory during training. No partial adapter checkpoints or automatic interrupted-training resume are configured for this short pilot; use a new RUN_NAME for a fresh attempt. Reexecuting model setup reloads the base model.
